#### 02.Station-Static Features
In this section, we will create station-specific features, save them to a small table, and later join it with other features for training.
We construct the following features,
* `t0_x_km`, `t0_y_km`: The relative location of a station in kilometers
* `t0_wet_ratio`: compared to the islandwise average, relative strength of rainfall (y_mm > 0mm) of a station in the next 30 minutes.
* `t0_heavy_ratio`: definition similar to the above, yet the threshold of rainfall replaced by y_mm > 5mm.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
MATRIX = ROOT / "task1_revision" / "data" / "matrix_5min"
OUT = ROOT / "task2" / "data"
assert (OUT / "labels").exists(), OUT

In [2]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .master("local[8]")                                     
         .appName("task2-station-static")
         .config("spark.driver.memory", "10g")                   
         .config("spark.sql.session.timeZone", "Asia/Singapore") 
         .config("spark.sql.shuffle.partitions", "96")           
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Read our processed data
labels = spark.read.parquet(str(OUT / "labels"))
labels.show(10)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 18:45:13 WARN Utils: Your hostname, Lynns-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.12 instead (on interface en0)
26/10/11 18:45:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 18:45:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------+------+-------------------+----+-----+---------+-------+----+
|station_id|  slot|                 ts|y_mm|label|past30_mm| regime|year|
+----------+------+-------------------+----+-----+---------+-------+----+
|      S108|420768|2021-01-01 00:00:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420769|2021-01-01 00:05:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420770|2021-01-01 00:10:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420771|2021-01-01 00:15:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420772|2021-01-01 00:20:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420773|2021-01-01 00:25:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420774|2021-01-01 00:30:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420775|2021-01-01 00:35:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420776|2021-01-01 00:40:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420777|2021-01-01 00:45:00| 0.0|  0.0|      0.0|dry_now|2021|
+----------+------+-------------------

In [3]:
# we will first construct the feature O/E ratio, representing the relative strength of rainfall 
# of a certain station compared to the islandwise average performance
tr = (labels.filter(F.col("year").between(2017, 2022))
            .withColumn("wet",   (F.col("label") > 0).cast("double"))
            .withColumn("heavy", (F.col("label") >= 6).cast("double")))   

net = tr.groupBy("slot").agg(F.avg("wet").alias("p_wet"),
                             F.avg("heavy").alias("p_heavy"))

# Here, we add a shrinkage term M=500 on both numerator and denominator, because the ratio O/E has a 
# extremely large span, applying shrinkage will help normalize stations with few observations 
M = 500
stn = (tr.join(net, on="slot")
         .groupBy("station_id")
         .agg(F.count("*").alias("n"),
              F.sum("wet").alias("O_wet"),     F.sum("p_wet").alias("E_wet"),
              F.sum("heavy").alias("O_heavy"), F.sum("p_heavy").alias("E_heavy"))
         .withColumn("t0_wet_ratio",   (F.col("O_wet") + M) / (F.col("E_wet") + M))
         .withColumn("t0_heavy_ratio", (F.col("O_heavy") + M) / (F.col("E_heavy") + M)))

In [4]:
pdf = stn.toPandas()
coords = pd.read_csv(MATRIX / "stations.csv")
pdf = pdf.merge(coords[["station_id", "location_longitude", "location_latitude"]], on="station_id")

LAT0, LON0 = 1.35, 103.82        
pdf["t0_x_km"] = (pdf.location_longitude - LON0) * 111.32 * np.cos(np.radians(LAT0))
pdf["t0_y_km"] = (pdf.location_latitude - LAT0) * 110.57

pdf.to_csv(OUT / "station_static.csv", index=False)

/Users/lynnchen/anaconda3/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [UNSUPPORTED_PACKAGE_VERSION] PyArrow >= 18.0.0 must be installed; however, your version is 16.1.0.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [5]:
pdf

,station_id,n,O_wet,E_wet,O_heavy,E_heavy,t0_wet_ratio,t0_heavy_ratio,location_longitude,location_latitude,t0_x_km,t0_y_km
0,S214,259365,15773.0,17057.170738,1865.0,2338.196415,0.926858,0.833276,103.88289,1.29911,6.998972,-5.626907
1,S11,458938,29381.0,27050.803732,3795.0,3586.910389,1.084578,1.050916,103.69380,1.37460,-14.044685,2.720022
2,S221,262992,17140.0,17273.514822,2436.0,2375.045044,0.992488,1.021201,103.89088,1.35691,7.888171,0.764039
3,S44,561096,35583.0,33671.657224,5047.0,4484.748053,1.055934,1.112794,103.68166,1.34583,-15.395734,-0.461077
4,S205,233535,14777.0,14935.663682,2032.0,2086.977743,0.989721,0.978748,103.91160,1.38829,10.194082,4.233725
...,...,...,...,...,...,...,...,...,...,...,...,...
82,S110,27482,1820.0,1854.306408,207.0,194.959949,0.985428,1.017325,103.86970,1.36061,5.531068,1.173148
83,S116,575950,31589.0,34327.063457,4344.0,4574.626739,0.921381,0.954553,103.75400,1.28100,-7.345081,-7.629330
84,S119,561412,32868.0,32569.600854,4358.0,4444.688433,1.009023,0.982468,103.86660,1.30105,5.186072,-5.412402
85,S210,258665,17319.0,17062.210472,2462.0,2349.858248,1.014622,1.039350,103.76904,1.44003,-5.671293,9.954617


Then, we construct time related features; we wrap it as a function so that it can be reused in both training and prediction sections. We decided to use sin/cos to encode time as it preserves the temporal dependency of hours within a day (23:00 and 00:00 has a gap numerically). 

Here we explain how the time features are generated in details. We first represent the date of a specific observation into a fraction of the entire year, so April 4 is converted to aorund 0.284. 

Then, we convert the timestamp in a day and in a year to be encoded by sin/cos functions. So h=16 under k=1, 2, 3 has sin=-0.866, 0.866, 0, denoted as t0_hod_s1, t0_hod_s2, t0_hod_s3. The combination of sin/cos can locate a specific point on the unit circle.

Finally, we classify each observation into three categories of seasonality, denoted by 0, 1, and 2.

In [6]:
import math
def add_time_features(df, ts_col="ts"):
    ts = F.col(ts_col)
    h = F.hour(ts) + F.minute(ts) / 60.0
    year = F.year(ts)
    days_in_year = F.datediff(F.make_date(year + 1, F.lit(1), F.lit(1)),
                              F.make_date(year, F.lit(1), F.lit(1)))
    d = (F.dayofyear(ts) - 1 + h / 24.0) / days_in_year

    for k in (1, 2, 3):
        angle = 2 * math.pi * k * h / 24                     
        df = (df.withColumn(f"t0_hod_s{k}", F.sin(angle))
                .withColumn(f"t0_hod_c{k}", F.cos(angle)))
    for k in (1, 2):
        angle = 2 * math.pi * k * d                      
        df = (df.withColumn(f"t0_doy_s{k}", F.sin(angle))
                .withColumn(f"t0_doy_c{k}", F.cos(angle)))

    month = F.month(ts)
    df = df.withColumn("t0_season",
                       F.when(month.isin(12, 1, 2, 3), 0.0)
                        .when(month.isin(6, 7, 8, 9), 2.0)
                        .otherwise(1.0))
    return df

In [7]:
# Use a small sample to check first
test = labels.filter((F.col("station_id") == "S07") & F.col("slot").isin(656247, 660864))
add_time_features(test).select("slot", "ts", *[c for c in add_time_features(test).columns if c.startswith("t0_")]).show(truncate=False)

+------+-------------------+-------------------+-------------------+------------------+-------------------+----------------------+---------+------------------+--------------------+--------------------+-------------------+---------+
|slot  |ts                 |t0_hod_s1          |t0_hod_c1          |t0_hod_s2         |t0_hod_c2          |t0_hod_s3             |t0_hod_c3|t0_doy_s1         |t0_doy_c1           |t0_doy_s2           |t0_doy_c2          |t0_season|
+------+-------------------+-------------------+-------------------+------------------+-------------------+----------------------+---------+------------------+--------------------+--------------------+-------------------+---------+
|660864|2023-04-14 16:00:00|-0.8660254037844384|-0.5000000000000004|0.8660254037844392|-0.4999999999999992|-4.898587196589413E-16|1.0      |0.9772437869053068|-0.21211926115979055|-0.41458446010269895|-0.9100108380900491|1.0      |
+------+-------------------+-------------------+-------------------+----

In [8]:
tf = add_time_features(labels)
tf.agg(F.min("t0_hod_s1"), F.max("t0_hod_s1"),
       F.max(F.col("t0_hod_s1")**2 + F.col("t0_hod_c1")**2 - 1)).show()     
tf.groupBy("t0_season").count().orderBy("t0_season").show()                

+--------------+--------------+------------------------------------------------------+
|min(t0_hod_s1)|max(t0_hod_s1)|max(((POWER(t0_hod_s1, 2) + POWER(t0_hod_c1, 2)) - 1))|
+--------------+--------------+------------------------------------------------------+
|          -1.0|           1.0|                                  2.220446049250313...|
+--------------+--------------+------------------------------------------------------+



+---------+--------+
|t0_season|   count|
+---------+--------+
|      0.0|15392890|
|      1.0|16073883|
|      2.0|16080785|
+---------+--------+

